# 7-6_2 Transformer モデル（分類タスク用）の今の書き方

`7-6_Transformer.ipynb` は、感情分析用の Transformer を Embedder、PositionalEncoder、Attention、FeedForward、TransformerBlock、ClassificationHead に分け、最後に `TransformerClassification` としてつなぐ。各層のあとに、IMDb のミニバッチを通してテンソルサイズを見ている。

このファイルは、層の定義は 7-6 と同じにする。ミニバッチと単語ベクトルの取り方だけ、`7-5_2_IMDb_Dataset_DataLoader.ipynb` の `Dataset` と `DataLoader` に置き換える。7-6 は `utils/dataloader.py` の `get_IMDb_DataLoaders_and_TEXT` を呼び、`batch.Text[0]` と `TEXT.vocab.vectors` を使っている。ここでは `text` と `vectors` をその位置に置く。バッチサイズは 7-6 と同じ 24 である。


# 7-6_2 学習目標

1.	Transformerのモジュール構成を理解する
2.	LSTMやRNNを使用せずCNNベースのTransformerで自然言語処理が可能な理由を理解する
3.	7-6と同じ各層を、今のDataLoaderが返すミニバッチで実装し、テンソルサイズを確認する


# 事前準備

7-5 で作った次のファイルを使う。

- `data/IMDb_train.tsv`
- `data/wiki-news-300d-1M.vec`

fastText の読み込みには gensim 3.8.3 を使う。パッケージは Docker の JupyterLab（http://127.0.0.1:8887/lab）に入っている。このファイルのテンソルサイズ確認では、訓練データのミニバッチだけを使う。


In [1]:
import math
import random
import re
import string
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from gensim.models import KeyedVectors
from torch.utils.data import DataLoader, Dataset


In [2]:
# Setup seeds
torch.manual_seed(1234)
np.random.seed(1234)
random.seed(1234)


# Dataset と DataLoader

7-6 は次の関数で、動作確認用のミニバッチと単語ベクトルを受け取っていた。

```python
from utils.dataloader import get_IMDb_DataLoaders_and_TEXT

train_dl, val_dl, test_dl, TEXT = get_IMDb_DataLoaders_and_TEXT(
    max_length=256, batch_size=24)

batch = next(iter(train_dl))
net1 = Embedder(TEXT.vocab.vectors)
x = batch.Text[0]
```

この関数の中身は `torchtext.data.Field` と `Iterator` である。`batch.Text[0]` は長さ 256 の単語 ID、`TEXT.vocab.vectors` は fastText を語彙の順に並べた行列である。`<pad>` の ID は 1 である。

ここでは 7-5_2 と同じ手順で、その 2 つを作る。tsv の前処理、80:20 の分割、語彙の並び、`<cls>` と `<eos>` を含む長さ 256 は 7-5 に合わせる。語彙は訓練データだけから作る。訓練データはシャッフルして取り出す。


In [3]:
def preprocessing_text(text):
    """7-6 が使う dataloader の preprocessing_text と同じ。"""
    text = re.sub("<br />", "", text)
    for p in string.punctuation:
        if (p == ".") or (p == ","):
            continue
        text = text.replace(p, " ")
    text = text.replace(".", " . ")
    text = text.replace(",", " , ")
    return text


def tokenizer_with_preprocessing(text):
    """7-6 が使う dataloader の tokenizer_with_preprocessing と同じ。"""
    text = preprocessing_text(text)
    return text.strip().split()


print(tokenizer_with_preprocessing("I like cats."))


['I', 'like', 'cats', '.']


In [4]:
class TextDataset(Dataset):
    """tsv の1行を (単語リスト, ラベル) として持つ。7-6 が使う TabularDataset に相当する。"""

    def __init__(self, samples):
        self.samples = list(samples)

    @classmethod
    def from_tsv(cls, path, tokenizer):
        samples = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                text, label, *_rest = line.rstrip("\n").split("\t")
                tokens = tokenizer(text)
                # Field は tokenize のあとで lower=True を各単語にかける
                tokens = [token.lower() for token in tokens]
                samples.append((tokens, label))
        return cls(samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


train_val_ds = TextDataset.from_tsv("./data/IMDb_train.tsv", tokenizer_with_preprocessing)
print("訓練および検証のデータ数", len(train_val_ds))


訓練および検証のデータ数 25000


In [5]:
def split_train_val(dataset, split_ratio=0.8, seed=1234):
    """7-6 が使う Dataset.split(split_ratio=0.8, random_state=random.seed(1234)) に相当する。"""
    random.seed(seed)
    order = random.sample(range(len(dataset)), len(dataset))
    train_len = int(round(split_ratio * len(dataset)))
    train_samples = [dataset[index] for index in order[:train_len]]
    val_samples = [dataset[index] for index in order[train_len:]]
    return TextDataset(train_samples), TextDataset(val_samples)


train_ds, val_ds = split_train_val(train_val_ds)
print("訓練データの数", len(train_ds))
print("検証データの数", len(val_ds))


訓練データの数 20000
検証データの数 5000


In [6]:
english_fasttext = KeyedVectors.load_word2vec_format(
    "data/wiki-news-300d-1M.vec", binary=False)


def build_vocab(dataset, min_freq=10):
    """7-6 が使う TEXT.build_vocab(train_ds, min_freq=10) に相当する。"""
    freqs = Counter()
    for tokens, _label in dataset:
        freqs.update(tokens)

    specials = ["<unk>", "<pad>", "<cls>", "<eos>"]
    for token in specials:
        if token in freqs:
            del freqs[token]

    words = [word for word, freq in freqs.items() if freq >= min_freq]
    words.sort(key=lambda word: (-freqs[word], word))
    itos = specials + words
    stoi = {word: index for index, word in enumerate(itos)}
    return stoi, itos


def build_vector_table(keyed_vectors, stoi):
    """7-6 の TEXT.vocab.vectors に相当する行列を作る。"""
    table = torch.zeros(len(stoi), keyed_vectors.vector_size)
    for word, index in stoi.items():
        if word in keyed_vectors:
            table[index] = torch.tensor(keyed_vectors[word].copy(), dtype=table.dtype)
    return table


stoi, itos = build_vocab(train_ds, min_freq=10)
vectors = build_vector_table(english_fasttext, stoi)

# 7-5 の実行例では torch.Size([17916, 300])
print("語彙ベクトルのサイズ:", vectors.shape)
print("<pad> の ID:", stoi["<pad>"])


語彙ベクトルのサイズ: torch.Size([17892, 300])
<pad> の ID: 1


In [7]:
max_length = 256
content_length = max_length - 2  # <cls> と <eos> の分を除いた本文の長さ
pad_id = stoi["<pad>"]  # 7-6 の input_pad = 1
unk_id = stoi["<unk>"]
cls_id = stoi["<cls>"]
eos_id = stoi["<eos>"]


def collate_batch(batch):
    """7-6 の batch.Text[0] に相当する単語 ID を返す。"""
    text_batch = []
    lengths = []
    labels = []

    for tokens, label in batch:
        ids = [stoi.get(token, unk_id) for token in tokens[:content_length]]
        ids = [cls_id] + ids + [eos_id]
        length = len(ids)
        ids += [pad_id] * (max_length - len(ids))

        text_batch.append(torch.tensor(ids, dtype=torch.long))
        lengths.append(length)
        labels.append(int(label))

    text = torch.stack(text_batch)
    lengths = torch.tensor(lengths, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)
    return text, lengths, labels


# 7-6 は batch_size=24。train=True はシャッフルする、という意味だった
train_dl = DataLoader(train_ds, batch_size=24, shuffle=True, collate_fn=collate_batch)

text, lengths, labels = next(iter(train_dl))
print("文章の ID:", text.shape)
print("長さ:", lengths.shape)
print("ラベル:", labels.shape)


文章の ID: torch.Size([24, 256])
長さ: torch.Size([24])
ラベル: torch.Size([24])


# Embedder

単語 ID を、学習済みベクトルの行として取り出す。`freeze=True` なので、この埋め込みは学習で更新しない。

7-6 は `Embedder(TEXT.vocab.vectors)` に `batch.Text[0]` を入れていた。ここでは `vectors` と、DataLoader が返した `text` を入れる。


In [8]:
class Embedder(nn.Module):
    '''idで示されている単語をベクトルに変換します'''

    def __init__(self, text_embedding_vectors):
        super(Embedder, self).__init__()

        self.embeddings = nn.Embedding.from_pretrained(
            embeddings=text_embedding_vectors, freeze=True)
        # freeze=Trueによりバックプロパゲーションで更新されず変化しなくなります

    def forward(self, x):
        x_vec = self.embeddings(x)

        return x_vec


In [9]:
# 動作確認

# ミニバッチの用意。7-6 の batch = next(iter(train_dl)); x = batch.Text[0]
x, lengths, labels = next(iter(train_dl))

# モデル構築。7-6 の Embedder(TEXT.vocab.vectors)
net1 = Embedder(vectors)

# 入出力
x1 = net1(x)  # 単語をベクトルに

print("入力のテンソルサイズ：", x.shape)
print("出力のテンソルサイズ：", x1.shape)


入力のテンソルサイズ： torch.Size([24, 256])
出力のテンソルサイズ： torch.Size([24, 256, 300])


# PositionalEncoder

単語ベクトルに、位置ごとの sin / cos を足す。系列の順番を、再帰や畳み込みなしで Attention に渡すための情報である。表 `pe` の長さは 7-6 と同じ 256 で、ミニバッチの次元を先頭に足してある。


In [10]:
class PositionalEncoder(nn.Module):
    '''入力された単語の位置を示すベクトル情報を付加する'''

    def __init__(self, d_model=300, max_seq_len=256):
        super().__init__()

        self.d_model = d_model  # 単語ベクトルの次元数

        # 単語の順番（pos）と埋め込みベクトルの次元の位置（i）によって一意に定まる値の表をpeとして作成
        pe = torch.zeros(max_seq_len, d_model)

        # GPUが使える場合はGPUへ送る、ここでは省略。実際に学習時には使用する
        # device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        # pe = pe.to(device)

        for pos in range(max_seq_len):
            for i in range(0, d_model, 2):
                pe[pos, i] = math.sin(pos / (10000 ** ((2 * i)/d_model)))
                
                # 誤植修正_200510 #79
                # pe[pos, i + 1] = math.cos(pos /
                #                          (10000 ** ((2 * (i + 1))/d_model)))
                pe[pos, i + 1] = math.cos(pos /
                                          (10000 ** ((2 * i)/d_model)))

        # 表peの先頭に、ミニバッチ次元となる次元を足す
        self.pe = pe.unsqueeze(0)

        # 勾配を計算しないようにする
        self.pe.requires_grad = False

    def forward(self, x):

        # 入力xとPositonal Encodingを足し算する
        # xがpeよりも小さいので、大きくする
        ret = math.sqrt(self.d_model)*x + self.pe
        return ret


In [11]:
# 動作確認

# モデル構築
net1 = Embedder(vectors)
net2 = PositionalEncoder(d_model=300, max_seq_len=256)

# 入出力
x, lengths, labels = next(iter(train_dl))
x1 = net1(x)  # 単語をベクトルに
x2 = net2(x1)

print("入力のテンソルサイズ：", x1.shape)
print("出力のテンソルサイズ：", x2.shape)


入力のテンソルサイズ： torch.Size([24, 256, 300])
出力のテンソルサイズ： torch.Size([24, 256, 300])


# Attention、FeedForward、TransformerBlock

Attention は、クエリ・キー・バリューを全結合層で変換した単一ヘッドである。`mask == 0` の位置、つまり `<pad>` は、softmax の前に大きな負の値にして重みを潰す。FeedForward は Attention の出力を全結合層 2 つで変換する。TransformerBlock は、層正規化、Attention、残差接続、層正規化、FeedForward、残差接続の順である。

7-6 のマスクは `(x != 1)` である。`<pad>` が 1 番だからで、ここでは `pad_id` を使う。


In [12]:
class Attention(nn.Module):
    '''Transformerは本当はマルチヘッドAttentionですが、
    分かりやすさを優先しシングルAttentionで実装します'''

    def __init__(self, d_model=300):
        super().__init__()

        # SAGANでは1dConvを使用したが、今回は全結合層で特徴量を変換する
        self.q_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)

        # 出力時に使用する全結合層
        self.out = nn.Linear(d_model, d_model)

        # Attentionの大きさ調整の変数
        self.d_k = d_model

    def forward(self, q, k, v, mask):
        # 全結合層で特徴量を変換
        k = self.k_linear(k)
        q = self.q_linear(q)
        v = self.v_linear(v)

        # Attentionの値を計算する
        # 各値を足し算すると大きくなりすぎるので、root(d_k)で割って調整
        weights = torch.matmul(q, k.transpose(1, 2)) / math.sqrt(self.d_k)

        # ここでmaskを計算
        mask = mask.unsqueeze(1)
        weights = weights.masked_fill(mask == 0, -1e9)

        # softmaxで規格化をする
        normlized_weights = F.softmax(weights, dim=-1)

        # AttentionをValueとかけ算
        output = torch.matmul(normlized_weights, v)

        # 全結合層で特徴量を変換
        output = self.out(output)

        return output, normlized_weights


In [13]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=1024, dropout=0.1):
        '''Attention層から出力を単純に全結合層2つで特徴量を変換するだけのユニットです'''
        super().__init__()

        self.linear_1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.linear_1(x)
        x = self.dropout(F.relu(x))
        x = self.linear_2(x)
        return x


In [14]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()

        # LayerNormalization層
        # https://pytorch.org/docs/stable/nn.html?highlight=layernorm
        self.norm_1 = nn.LayerNorm(d_model)
        self.norm_2 = nn.LayerNorm(d_model)

        # Attention層
        self.attn = Attention(d_model)

        # Attentionのあとの全結合層2つ
        self.ff = FeedForward(d_model)

        # Dropout
        self.dropout_1 = nn.Dropout(dropout)
        self.dropout_2 = nn.Dropout(dropout)

    def forward(self, x, mask):
        # 正規化とAttention
        x_normlized = self.norm_1(x)
        output, normlized_weights = self.attn(
            x_normlized, x_normlized, x_normlized, mask)
        
        x2 = x + self.dropout_1(output)

        # 正規化と全結合層
        x_normlized2 = self.norm_2(x2)
        output = x2 + self.dropout_2(self.ff(x_normlized2))

        return output, normlized_weights


In [15]:
# 動作確認

# モデル構築
net1 = Embedder(vectors)
net2 = PositionalEncoder(d_model=300, max_seq_len=256)
net3 = TransformerBlock(d_model=300)

# maskの作成
x, lengths, labels = next(iter(train_dl))
input_mask = (x != pad_id)
print(input_mask[0])

# 入出力
x1 = net1(x)  # 単語をベクトルに
x2 = net2(x1)  # Positon情報を足し算
x3, normlized_weights = net3(x2, input_mask)  # Self-Attentionで特徴量を変換

print("入力のテンソルサイズ：", x2.shape)
print("出力のテンソルサイズ：", x3.shape)
print("Attentionのサイズ：", normlized_weights.shape)


tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, Tr

# ClassificationHead

各文の先頭、つまり `<cls>` の特徴量だけを取り、positive / negative の 2 値に写す。


In [16]:
class ClassificationHead(nn.Module):
    '''Transformer_Blockの出力を使用し、最後にクラス分類させる'''

    def __init__(self, d_model=300, output_dim=2):
        super().__init__()

        # 全結合層
        self.linear = nn.Linear(d_model, output_dim)  # output_dimはポジ・ネガの2つ

        # 重み初期化処理
        nn.init.normal_(self.linear.weight, std=0.02)
        nn.init.normal_(self.linear.bias, 0)

    def forward(self, x):
        x0 = x[:, 0, :]  # 各ミニバッチの各文の先頭の単語の特徴量（300次元）を取り出す
        out = self.linear(x0)

        return out


In [17]:
# 動作確認

# ミニバッチの用意
x, lengths, labels = next(iter(train_dl))

# モデル構築
net1 = Embedder(vectors)
net2 = PositionalEncoder(d_model=300, max_seq_len=256)
net3 = TransformerBlock(d_model=300)
net4 = ClassificationHead(output_dim=2, d_model=300)

# 入出力
# 7-6 は前のセルの input_mask を続けて使っている。取り直したミニバッチに合わせて作り直す
input_mask = (x != pad_id)
x1 = net1(x)  # 単語をベクトルに
x2 = net2(x1)  # Positon情報を足し算
x3, normlized_weights = net3(x2, input_mask)  # Self-Attentionで特徴量を変換
x4 = net4(x3)  # 最終出力の0単語目を使用して、分類0-1のスカラーを出力

print("入力のテンソルサイズ：", x3.shape)
print("出力のテンソルサイズ：", x4.shape)


入力のテンソルサイズ： torch.Size([24, 256, 300])
出力のテンソルサイズ： torch.Size([24, 2])


# TransformerClassification

上の層を、埋め込み、位置エンコーディング、TransformerBlock 2 段、分類ヘッドの順につなぐ。Block は 2 段なので、Attention の重みも 2 つ返る。


In [18]:
# 最終的なTransformerモデルのクラス


class TransformerClassification(nn.Module):
    '''Transformerでクラス分類させる'''

    def __init__(self, text_embedding_vectors, d_model=300, max_seq_len=256, output_dim=2):
        super().__init__()

        # モデル構築
        self.net1 = Embedder(text_embedding_vectors)
        self.net2 = PositionalEncoder(d_model=d_model, max_seq_len=max_seq_len)
        self.net3_1 = TransformerBlock(d_model=d_model)
        self.net3_2 = TransformerBlock(d_model=d_model)
        self.net4 = ClassificationHead(output_dim=output_dim, d_model=d_model)

    def forward(self, x, mask):
        x1 = self.net1(x)  # 単語をベクトルに
        x2 = self.net2(x1)  # Positon情報を足し算
        x3_1, normlized_weights_1 = self.net3_1(
            x2, mask)  # Self-Attentionで特徴量を変換
        x3_2, normlized_weights_2 = self.net3_2(
            x3_1, mask)  # Self-Attentionで特徴量を変換
        x4 = self.net4(x3_2)  # 最終出力の0単語目を使用して、分類0-1のスカラーを出力
        return x4, normlized_weights_1, normlized_weights_2


In [19]:
# 動作確認

# ミニバッチの用意
x, lengths, labels = next(iter(train_dl))

# モデル構築
net = TransformerClassification(
    text_embedding_vectors=vectors, d_model=300, max_seq_len=256, output_dim=2)

# 入出力
input_mask = (x != pad_id)
out, normlized_weights_1, normlized_weights_2 = net(x, input_mask)

print("出力のテンソルサイズ：", out.shape)
print("出力テンソルのsigmoid：", F.softmax(out, dim=1))


出力のテンソルサイズ： torch.Size([24, 2])
出力テンソルのsigmoid： tensor([[0.3155, 0.6845],
        [0.2872, 0.7128],
        [0.3263, 0.6737],
        [0.3311, 0.6689],
        [0.3389, 0.6611],
        [0.2901, 0.7099],
        [0.3031, 0.6969],
        [0.2617, 0.7383],
        [0.2910, 0.7090],
        [0.2846, 0.7154],
        [0.3473, 0.6527],
        [0.3422, 0.6578],
        [0.3298, 0.6702],
        [0.3081, 0.6919],
        [0.3175, 0.6825],
        [0.3424, 0.6576],
        [0.3091, 0.6909],
        [0.3018, 0.6982],
        [0.2917, 0.7083],
        [0.3077, 0.6923],
        [0.3441, 0.6559],
        [0.3294, 0.6706],
        [0.2891, 0.7109],
        [0.3049, 0.6951]], grad_fn=<SoftmaxBackward>)


層の定義は 7-6 と同じである。`utils/transformer.py` では、PositionalEncoder が位置の表を使うデバイスへ送る。このファイルは 7-6 に合わせ、その処理をコメントのままにしている。7-7 と `7-7_2_transformer_training_inference.ipynb` は、`transformer.py` から `TransformerClassification` を読み込む。このファイルは各層の入出力を見るためのもので、`transformer.py` は書き換えない。

以上
